# DINO Pretraining — TCIA Brain-Met WSI Encoder (Arm A)

**Project:** Pretraining Strategy Matters: Benchmarking WSI Encoders for Brain Metastasis Risk Prediction in NSCLC  
**Encoder:** ViT-Tiny/16, warm-started from ImageNet weights  
**Data:** 96 TCIA brain-met NSCLC WSIs, 388,681 tiles (Vahadane normalized)  
**Target:** 10 epochs, checkpoint every epoch, resumable across sessions

### Design choices
- ViT-Tiny used (not ViT-Small) — justified by small dataset size (96 WSIs). Large model capacity risks memorizing slide-specific features.
- Vahadane stain normalization already baked into h5 tiles — NO on-the-fly Macenko needed
- HED stain jitter (Tellez et al., TMI 2019)
- HistoRotate 360° (PathDino, CVPR 2024)
- Mixed precision (fp16) for memory efficiency
- Prefetch chunk loader — loads next chunk while GPU trains on current
- Single checkpoint file — overwrites every epoch, never runs out of disk

### Run order
**Cell 0 → restart runtime → Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10**

### To resume from checkpoint
Set `RESUME_CKPT` in Cell 2 to the path of `dino_latest.pt` in your Drive.

In [2]:
# ── CELL 0 — Fix environment (run once, then restart runtime) ──────────────
import subprocess
subprocess.run(['pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run(['pip', 'install', '-q', 'timm==0.9.16'], check=True)
subprocess.run(['pip', 'install', '-q', 'staintools', 'spams-bin', 'h5py'], check=True)
print('Done. Go to Runtime → Restart session, then run from Cell 1.')

Done. Go to Runtime → Restart session, then run from Cell 1.


In [3]:
# ── CELL 1 — Mount Google Drive ────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.


In [4]:
# ── CELL 2 — Imports and Config ────────────────────────────────────────────
import os, sys, time, math, random, glob, json, gc
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import h5py
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import timm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, {p.total_memory/1e9:.1f} GB')

# ── PATHS — update if your Drive folder is different ──────────────────────
H5_DIR      = '/content/drive/MyDrive/mtech_proj/brain-histo-tiles'
CKPT_DIR    = Path('/content/drive/MyDrive/mtech_proj/checkpoints')
RESUME_CKPT = None
# To resume: RESUME_CKPT = '/content/drive/MyDrive/mtech_proj/checkpoints/dino_latest.pt'
# ──────────────────────────────────────────────────────────────────────────

CKPT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    'arch'                : 'vit_tiny',   # ViT-Tiny — justified for 96-slide dataset
    'patch_size'          : 16,
    'out_dim'             : 4096,         # smaller head for smaller model
    'norm_last_layer'     : True,
    'use_bn_in_head'      : False,
    'n_epochs'            : 10,           # 10 epochs sufficient for 96 slides
    'batch_size'          : 256,          # ViT-Tiny is small — fits large batch
    'num_workers'         : 4,
    'img_size'            : 224,
    'lr'                  : 5e-4,
    'min_lr'              : 1e-6,
    'weight_decay_start'  : 0.04,
    'weight_decay_end'    : 0.4,
    'warmup_epochs'       : 3,
    'momentum_teacher'    : 0.996,
    'momentum_teacher_end': 1.0,
    'teacher_temp'        : 0.04,
    'student_temp'        : 0.1,
    'teacher_temp_warmup' : 4,
    'center_momentum'     : 0.9,
    'global_crops_scale'  : (0.4, 1.0),
    'local_crops_scale'   : (0.05, 0.4),
    'local_crops_number'  : 2,            # 2 local crops — fast + memory efficient
    'local_crop_size'     : 96,
    'slides_per_chunk'    : 10,           # 10 slides/chunk — good balance
    'session_budget_hrs'  : 11.5,         # Colab typically disconnects ~12h
}

print(f'Config loaded. arch={CFG["arch"]} | n_epochs={CFG["n_epochs"]} | batch={CFG["batch_size"]}')

Device: cuda
GPU: NVIDIA A100-SXM4-40GB, 42.4 GB
Config loaded. arch=vit_tiny | n_epochs=10 | batch=256


In [5]:
# ── CELL 3 — Collect H5 Files ──────────────────────────────────────────────
all_h5 = sorted(glob.glob(os.path.join(H5_DIR, '*.h5')))
print(f'Total h5 files: {len(all_h5)}')
assert len(all_h5) > 0, f'No h5 files found in {H5_DIR}'

with h5py.File(all_h5[0], 'r') as f:
    key   = list(f.keys())[0]
    shape = f[key].shape
    print(f'Sample: key={key}, shape={shape}')

n_chunks_per_epoch = math.ceil(len(all_h5) / CFG['slides_per_chunk'])
print(f'Chunks per epoch: {n_chunks_per_epoch}')

Total h5 files: 96
Sample: key=coords, shape=(4650, 2)
Chunks per epoch: 10


In [6]:
# ── CELL 4 — Augmentations ─────────────────────────────────────────────────
# No Macenko — tiles already Vahadane normalized during preprocessing

class HEDJitter:
    """Stochastic HED stain perturbation (Tellez et al., TMI 2019)."""
    def __init__(self, theta=0.05):
        self.theta = theta

    def __call__(self, img):
        img = img.convert('RGB')
        arr = np.array(img).astype(np.float32) / 255.0
        hed = self._rgb2hed(arr)
        for i in range(3):
            alpha = np.random.uniform(1 - self.theta, 1 + self.theta)
            beta  = np.random.uniform(-self.theta, self.theta)
            hed[:, :, i] = hed[:, :, i] * alpha + beta
        rgb = np.clip(self._hed2rgb(hed) * 255, 0, 255).astype(np.uint8)
        return Image.fromarray(rgb)

    @staticmethod
    def _rgb2hed(rgb):
        rgb = np.maximum(rgb, 1e-6)
        if rgb.ndim == 2:
            rgb = np.stack([rgb, rgb, rgb], axis=-1)
        elif rgb.shape[-1] == 1:
            rgb = np.repeat(rgb, 3, axis=-1)
        M = np.array([[0.6500286, 0.7044536, 0.2860126],
                      [0.7044522, 0.6505834, 0.2860136],
                      [0.2859801, 0.2860286, 0.9144722]])
        return (-np.log(rgb)) @ np.linalg.inv(M).T

    @staticmethod
    def _hed2rgb(hed):
        M = np.array([[0.6500286, 0.7044536, 0.2860126],
                      [0.7044522, 0.6505834, 0.2860136],
                      [0.2859801, 0.2860286, 0.9144722]])
        return np.exp(-(hed @ M.T))


class HistoRotate:
    """Full 360 degree random rotation (Alfasly et al., CVPR 2024)."""
    def __call__(self, img):
        return TF.rotate(img, random.uniform(0, 360))


MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def make_global_tf(cfg):
    return T.Compose([
        HEDJitter(0.05),
        T.RandomResizedCrop(cfg['img_size'], scale=cfg['global_crops_scale'],
                            interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        HistoRotate(),
        T.ColorJitter(0.4, 0.4, 0.2, 0.1),
        T.RandomGrayscale(p=0.2),
        T.GaussianBlur(11, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
        T.RandomErasing(p=0.25),
    ])

def make_local_tf(cfg):
    return T.Compose([
        HEDJitter(0.05),
        T.RandomResizedCrop(cfg['local_crop_size'], scale=cfg['local_crops_scale'],
                            interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        HistoRotate(),
        T.ColorJitter(0.4, 0.4, 0.2, 0.1),
        T.RandomGrayscale(p=0.2),
        T.GaussianBlur(9, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])

global_tf = make_global_tf(CFG)
local_tf  = make_local_tf(CFG)
print('Transforms ready. No Macenko — tiles already Vahadane normalized.')

Transforms ready. No Macenko — tiles already Vahadane normalized.


In [7]:
# ── CELL 5 — Chunk Dataset with Prefetching ────────────────────────────────

def load_h5_tiles(path):
    """Returns (N, H, W, 3) uint8 array."""
    with h5py.File(path, 'r') as f:
        key = list(f.keys())[0]
        arr = f[key][:]
    if arr.ndim == 4 and arr.shape[1] == 3:
        arr = arr.transpose(0, 2, 3, 1)
    return arr.astype(np.uint8)


class PrefetchChunkLoader:
    """Loads next chunk in background while GPU trains on current chunk."""
    def __init__(self, h5_paths):
        self.h5_paths  = h5_paths
        self._executor = ThreadPoolExecutor(max_workers=1)
        self._future   = None

    def prefetch(self, indices):
        self._future = self._executor.submit(self._load, indices)

    def get(self):
        if self._future is not None:
            result = self._future.result()
            self._future = None
            return result
        return np.zeros((0, 256, 256, 3), dtype=np.uint8)

    def _load(self, indices):
        parts = []
        for i in indices:
            try:
                parts.append(load_h5_tiles(self.h5_paths[i]))
            except Exception as e:
                print(f'  Skip {os.path.basename(self.h5_paths[i])}: {e}')
        return np.concatenate(parts, axis=0) if parts else \
               np.zeros((0, 256, 256, 3), dtype=np.uint8)


class DINOChunkDataset(Dataset):
    def __init__(self, h5_paths, global_tf, local_tf, n_local):
        self.h5_paths  = h5_paths
        self.global_tf = global_tf
        self.local_tf  = local_tf
        self.n_local   = n_local
        self.tiles     = np.zeros((0, 256, 256, 3), dtype=np.uint8)

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):
        img = Image.fromarray(self.tiles[idx])
        return [self.global_tf(img), self.global_tf(img)] + \
               [self.local_tf(img) for _ in range(self.n_local)]


def dino_collate(batch):
    n = len(batch[0])
    return [torch.stack([b[i] for b in batch]) for i in range(n)]


dataset    = DINOChunkDataset(all_h5, global_tf, local_tf, CFG['local_crops_number'])
prefetcher = PrefetchChunkLoader(all_h5)
print(f'Dataset ready. slides_per_chunk={CFG["slides_per_chunk"]}')

Dataset ready. slides_per_chunk=10


In [8]:
# ── CELL 6 — DINO Model (ViT-Tiny) ────────────────────────────────────────

class DINOHead(nn.Module):
    def __init__(self, in_dim, out_dim, use_bn=False, norm_last_layer=True,
                 hidden_dim=512, bottleneck_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim), nn.GELU(),
            nn.Linear(hidden_dim, bottleneck_dim),
        )
        self.last_layer = nn.utils.weight_norm(
            nn.Linear(bottleneck_dim, out_dim, bias=False))
        self.last_layer.weight_g.data.fill_(1)
        if norm_last_layer:
            self.last_layer.weight_g.requires_grad = False

    def forward(self, x):
        x = self.mlp(x)
        x = F.normalize(x, dim=-1)
        return self.last_layer(x)


class DINONet(nn.Module):
    def __init__(self, backbone, head):
        super().__init__()
        self.backbone = backbone
        self.head     = head

    def forward(self, x):
        return self.head(self.backbone(x))


# ViT-Tiny warm-started from ImageNet weights
bb_s = timm.create_model('vit_tiny_patch16_224.augreg_in21k',
                          pretrained=True, num_classes=0)
bb_t = timm.create_model('vit_tiny_patch16_224.augreg_in21k',
                          pretrained=True, num_classes=0)
embed_dim = bb_s.num_features
print(f'ViT-Tiny embed_dim: {embed_dim}')

student = DINONet(bb_s, DINOHead(embed_dim, CFG['out_dim'],
                                  CFG['use_bn_in_head'], CFG['norm_last_layer']))
teacher = DINONet(bb_t, DINOHead(embed_dim, CFG['out_dim'],
                                  CFG['use_bn_in_head'], CFG['norm_last_layer']))

for p in teacher.parameters():
    p.requires_grad = False
teacher.load_state_dict(student.state_dict())

student = student.to(device)
teacher = teacher.to(device)

print(f'Student params: {sum(p.numel() for p in student.parameters()):,}')
print(f'Teacher params: {sum(p.numel() for p in teacher.parameters()):,}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


ViT-Tiny embed_dim: 192


/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Student params: 6,479,936
Teacher params: 6,479,936


In [9]:
# ── CELL 7 — DINO Loss ─────────────────────────────────────────────────────

class DINOLoss(nn.Module):
    def __init__(self, out_dim, n_crops, warmup_teacher_temp, teacher_temp,
                 warmup_epochs, n_epochs, student_temp, center_momentum):
        super().__init__()
        self.student_temp    = student_temp
        self.center_momentum = center_momentum
        self.n_crops         = n_crops
        self.register_buffer('center', torch.zeros(1, out_dim))
        self.teacher_temp_schedule = np.concatenate([
            np.linspace(warmup_teacher_temp, teacher_temp, warmup_epochs),
            np.full(n_epochs - warmup_epochs, teacher_temp),
        ])

    def forward(self, student_out, teacher_out, epoch):
        s    = (student_out / self.student_temp).chunk(self.n_crops)
        temp = self.teacher_temp_schedule[
                   min(epoch, len(self.teacher_temp_schedule) - 1)]
        t    = F.softmax((teacher_out - self.center) / temp,
                         dim=-1).detach().chunk(2)
        loss, n = 0.0, 0
        for iq, q in enumerate(t):
            for iv, v in enumerate(s):
                if iv == iq:
                    continue
                loss += torch.sum(-q * F.log_softmax(v, dim=-1), dim=-1).mean()
                n    += 1
        loss /= n
        self.update_center(t)
        return loss

    @torch.no_grad()
    def update_center(self, t):
        bc = torch.cat(t).mean(0, keepdim=True)
        self.center = (self.center * self.center_momentum
                       + bc * (1 - self.center_momentum))


n_crops   = 2 + CFG['local_crops_number']
dino_loss = DINOLoss(
    out_dim             = CFG['out_dim'],
    n_crops             = n_crops,
    warmup_teacher_temp = 0.04,               # ramp from 0.04 → teacher_temp
    teacher_temp        = CFG['teacher_temp'],  # 0.04 final (or raise to 0.07 for sharper)
    warmup_epochs       = CFG['teacher_temp_warmup'],
    n_epochs            = CFG['n_epochs'],
    student_temp        = CFG['student_temp'],
    center_momentum     = CFG['center_momentum'],
).to(device)
print(f'DINO loss ready. n_crops={n_crops}')

DINO loss ready. n_crops=4


In [10]:
# ── CELL 8 — Optimizer and Schedulers ─────────────────────────────────────

optimizer = torch.optim.AdamW(student.parameters(), lr=CFG['lr'],
                               weight_decay=CFG['weight_decay_start'])

def get_lr(epoch):
    if epoch < CFG['warmup_epochs']:
        return CFG['lr'] * max(epoch, 1) / CFG['warmup_epochs']
    p = (epoch - CFG['warmup_epochs']) / max(CFG['n_epochs'] - CFG['warmup_epochs'], 1)
    return CFG['min_lr'] + 0.5 * (CFG['lr'] - CFG['min_lr']) * (1 + math.cos(math.pi * p))

def get_wd(epoch):
    p = epoch / CFG['n_epochs']
    return CFG['weight_decay_start'] + (CFG['weight_decay_end'] - CFG['weight_decay_start']) * p

def get_momentum(epoch):
    p = epoch / CFG['n_epochs']
    return CFG['momentum_teacher'] + (CFG['momentum_teacher_end'] - CFG['momentum_teacher']) * p

print('Optimizer ready.')

Optimizer ready.


In [11]:
# ── CELL 9 — Resume from Checkpoint ───────────────────────────────────────

start_epoch  = 0
loss_history = []

if RESUME_CKPT and os.path.exists(RESUME_CKPT):
    print(f'Loading checkpoint: {RESUME_CKPT}')
    ckpt  = torch.load(RESUME_CKPT, map_location=device)
    s_mod = student.module if isinstance(student, nn.DataParallel) else student
    t_mod = teacher.module if isinstance(teacher, nn.DataParallel) else teacher
    s_mod.load_state_dict(ckpt['student'])
    t_mod.load_state_dict(ckpt['teacher'])
    optimizer.load_state_dict(ckpt['optimizer'])
    dino_loss.load_state_dict(ckpt['dino_loss'])
    start_epoch  = ckpt['epoch'] + 1
    loss_history = ckpt.get('loss_history', [])
    print(f'Resumed from epoch {ckpt["epoch"]}. Continuing at epoch {start_epoch}.')
else:
    print(f'Fresh start. Training epochs 0 to {CFG["n_epochs"]-1}.')

Fresh start. Training epochs 0 to 9.


In [ ]:
# ── CELL 10 — Training Loop ────────────────────────────────────────────────
import os, shutil
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
torch.cuda.empty_cache()

# ── Copy h5 files from Drive to local disk (10-20x faster reads) ──────────
LOCAL_H5 = '/content/h5_tiles'
os.makedirs(LOCAL_H5, exist_ok=True)
print('Copying h5 files to local disk...')
copy_t0 = time.time()
for i, src in enumerate(all_h5):
    dst = os.path.join(LOCAL_H5, os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
    print(f'  [{i+1}/{len(all_h5)}] {os.path.basename(src)}', flush=True)
local_h5 = sorted(glob.glob(os.path.join(LOCAL_H5, '*.h5')))
prefetcher.h5_paths = local_h5
dataset.h5_paths    = local_h5
print(f'Copy done in {(time.time()-copy_t0)/60:.1f} min. {len(local_h5)} files on local disk.')
# ──────────────────────────────────────────────────────────────────────────

scaler = torch.cuda.amp.GradScaler()

SESSION_START = time.time()
BUDGET_SECS   = CFG['session_budget_hrs'] * 3600

print(f'Training epochs {start_epoch} to {CFG["n_epochs"]-1}')
print(f'Session budget: {CFG["session_budget_hrs"]}h | Checkpoints: {CKPT_DIR}')
print('=' * 70)

for epoch in range(start_epoch, CFG['n_epochs']):

    if time.time() - SESSION_START > BUDGET_SECS:
        print(f'Session budget reached before epoch {epoch}. Stopping.')
        print(f'Checkpoint saved at: {CKPT_DIR}/dino_latest.pt')
        break

    epoch_t0 = time.time()

    lr = get_lr(epoch)
    wd = get_wd(epoch)
    for pg in optimizer.param_groups:
        pg['lr'] = lr
        pg['weight_decay'] = wd
    momentum = get_momentum(epoch)

    slide_idx = list(range(len(local_h5)))
    random.shuffle(slide_idx)
    n_chunks  = math.ceil(len(slide_idx) / CFG['slides_per_chunk'])

    epoch_loss = 0.0
    epoch_n    = 0
    student.train()

    prefetcher.prefetch(slide_idx[:CFG['slides_per_chunk']])

    for ci in range(n_chunks):

        chunk_t0 = time.time()
        tiles    = prefetcher.get()
        load_sec = time.time() - chunk_t0

        next_start = (ci + 1) * CFG['slides_per_chunk']
        next_chunk = slide_idx[next_start:next_start + CFG['slides_per_chunk']]
        if next_chunk:
            prefetcher.prefetch(next_chunk)

        dataset.tiles = tiles
        if len(dataset) == 0:
            continue

        print(f'  Ep {epoch} | Chunk {ci+1}/{n_chunks} | '
              f'{len(dataset)} tiles | load={load_sec:.1f}s', flush=True)

        loader = DataLoader(
            dataset,
            batch_size      = CFG['batch_size'],
            shuffle         = True,
            num_workers     = CFG['num_workers'],
            pin_memory      = True,
            drop_last       = True,
            collate_fn      = dino_collate,
            prefetch_factor = 2,
        )

        for views in loader:
            views = [v.to(device, non_blocking=True) for v in views]

            with torch.cuda.amp.autocast():
                global_views = torch.cat(views[:2], dim=0)
                local_views  = torch.cat(views[2:], dim=0)

                student_global = student(global_views)
                local_resized  = F.interpolate(
                    local_views, size=224,
                    mode='bicubic', align_corners=False)
                student_local  = student(local_resized)
                student_out    = torch.cat([student_global, student_local], dim=0)

                with torch.no_grad():
                    teacher_out = teacher(global_views)

                loss = dino_loss(student_out, teacher_out, epoch)

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(student.parameters(), 3.0)
            scaler.step(optimizer)
            scaler.update()

            with torch.no_grad():
                s_params = (student.module.parameters()
                            if isinstance(student, nn.DataParallel)
                            else student.parameters())
                t_params = (teacher.module.parameters()
                            if isinstance(teacher, nn.DataParallel)
                            else teacher.parameters())
                for sp, tp in zip(s_params, t_params):
                    tp.data.mul_(momentum).add_((1 - momentum) * sp.data)

            epoch_loss += loss.item()
            epoch_n    += 1

        chunk_total = time.time() - chunk_t0
        print(f'  Chunk {ci+1} done | train={chunk_total-load_sec:.1f}s | '
              f'total={chunk_total:.1f}s', flush=True)

        del loader
        gc.collect()

    avg_loss   = epoch_loss / max(epoch_n, 1)
    epoch_time = time.time() - epoch_t0
    elapsed    = time.time() - SESSION_START

    loss_history.append({
        'epoch'    : epoch,
        'loss'     : avg_loss,
        'time_min' : epoch_time / 60
    })

    print(f'Ep {epoch:3d}/{CFG["n_epochs"]-1} | loss={avg_loss:.4f} | '
          f'batches={epoch_n} | {epoch_time/60:.1f}m/ep | '
          f'lr={lr:.1e} | elapsed={elapsed/3600:.2f}h')

    s_sd = (student.module.state_dict()
            if isinstance(student, nn.DataParallel)
            else student.state_dict())
    t_sd = (teacher.module.state_dict()
            if isinstance(teacher, nn.DataParallel)
            else teacher.state_dict())
    ckpt = dict(epoch=epoch, student=s_sd, teacher=t_sd,
                optimizer=optimizer.state_dict(),
                dino_loss=dino_loss.state_dict(),
                loss_history=loss_history, cfg=CFG)
    torch.save(ckpt, CKPT_DIR / 'dino_latest.pt')
    print(f'  Checkpoint saved. epoch={epoch}')

print('\nTraining done.')

Copying h5 files to local disk...
  [1/96] YG_0PGQQ6USQ9JB_wsi.h5
  [2/96] YG_2I5MDHB0AXEA_wsi.h5
  [3/96] YG_2VWCV5YWB078_wsi.h5
  [4/96] YG_30TUKBI1ZXBK_wsi.h5
  [5/96] YG_31S9L6RD6RCA_wsi.h5
  [6/96] YG_37RLQEBG98MP_wsi.h5
  [7/96] YG_38SD26C9HLLT_wsi.h5
  [8/96] YG_3OAF908JG3XG_wsi.h5
  [9/96] YG_3ULZIC6OE5NB_wsi.h5
  [10/96] YG_3YJ63A56N6VQ_wsi.h5
  [11/96] YG_4RD15Z2MNGTF_wsi.h5
  [12/96] YG_5WXJER534E8W_wsi.h5
  [13/96] YG_62XGKPMBQUTH_wsi.h5
  [14/96] YG_6NJKER3GCTC9_wsi.h5


In [ ]:
# ── CELL 11 — Loss Curve ───────────────────────────────────────────────────

if loss_history:
    ep = [d['epoch'] for d in loss_history]
    ls = [d['loss']  for d in loss_history]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(ep, ls, 'b-o', markersize=5, linewidth=1.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('DINO Loss')
    ax.set_title('DINO Pretraining — TCIA Brain-Met Encoder (Arm A) — ViT-Tiny')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(CKPT_DIR / 'dino_loss_curve.png'), dpi=150)
    plt.show()
    print(f'Best loss: {min(ls):.4f} at epoch {ep[ls.index(min(ls))]}')
    print(f'Avg time/epoch: {sum(d["time_min"] for d in loss_history)/len(loss_history):.1f} min')
else:
    print('No history yet.')

In [ ]:
# ── CELL 12 — Save Final Encoder (backbone only) ───────────────────────────
# Run after all epochs complete.
# Saves backbone only — no projection head — ready for feature extraction.

ckpt_loaded = torch.load(str(CKPT_DIR / 'dino_latest.pt'), map_location='cpu')

# Extract backbone weights only
full_sd = ckpt_loaded['student']
bb_sd   = {k.replace('backbone.', ''): v
           for k, v in full_sd.items() if k.startswith('backbone.')}

encoder_save = {
    'backbone_state_dict': bb_sd,
    'arch'               : CFG['arch'],
    'patch_size'         : CFG['patch_size'],
    'embed_dim'          : embed_dim,
    'epochs_trained'     : ckpt_loaded['epoch'],
    'final_loss'         : ckpt_loaded['loss_history'][-1]['loss'],
    'dataset'            : 'TCIA Brain-Mets-Lung (96 WSIs, 388681 tiles, Vahadane normalized)',
    'notes'              : 'ViT-Tiny/16 DINO, warm-start ImageNet augreg_in21k, HEDJitter+HistoRotate360',
}

save_path = str(CKPT_DIR / 'tcia_dino_encoder.pt')
torch.save(encoder_save, save_path)
print(f'Saved: {save_path}')
print(f'Epochs trained: {encoder_save["epochs_trained"]}')
print(f'Final loss    : {encoder_save["final_loss"]:.4f}')
print(f'Embed dim     : {embed_dim}')

In [ ]:
# ── CELL 13 — Sanity Check ─────────────────────────────────────────────────

enc = timm.create_model('vit_tiny_patch16_224.augreg_in21k',
                         pretrained=False, num_classes=0)
enc.load_state_dict(
    torch.load(str(CKPT_DIR / 'tcia_dino_encoder.pt'),
               map_location='cpu')['backbone_state_dict'])
enc.eval()

with torch.no_grad():
    out = enc(torch.randn(4, 3, 224, 224))

print(f'Output shape: {out.shape}')  # expected (4, 192)
assert out.shape == (4, 192), f'Expected (4, 192), got {out.shape}'
print('Encoder OK. Ready for feature extraction.')